In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.3 PCA, Covariance, and Whitening

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter IV — The Singular Value Decomposition",
    number="4.3",
    title="PCA, Covariance, and Whitening",
    blurb="Principal component analysis is the SVD of a centred data matrix — "
    "nothing more — and every one of its promises is an identity from the last "
    "two notebooks wearing statistical clothes. Forget the centring, though, "
    "and the first component quietly points at the mean.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

Principal component analysis is the most widely used algorithm this course
will meet, and this notebook's claim is that there is nothing in it beyond
[§4.1](svd-geometry.ipynb) and [§4.2](low-rank-eckart-young.ipynb): PCA *is*
the SVD of a centred data matrix. The principal directions are right singular
vectors, the variances are $\sigma_i^2/(n-1)$, the "explained variance" is
Frobenius energy, and the reconstruction guarantee is Eckart–Young verbatim.
Every statistical promise is a linear-algebra identity already measured, and
this notebook re-measures each in its new costume — down to agreement with
`sklearn.decomposition.PCA` at $3\times10^{-16}$, which is to say: sklearn
computes the same SVD.

Two things here are genuinely new. **Whitening** runs PCA backwards, using
$C^{-1/2}$ to manufacture uncorrelated unit-variance coordinates — the inverse
of the trick that *generated* correlated data in
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb). And the notebook
closes with two cautionary measurements: what happens when the centring is
skipped (the first component swings to point at the mean, cosine $0.99998$),
and how the PCA line differs from the regression line — same cloud, two
different questions, two measurably different answers.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Strang {cite}`strang2019learning` Chapter I.9; Hastie, Tibshirani
> and Friedman {cite}`hastie2009` §14.5; Deisenroth, Faisal and Ong
> {cite}`deisenroth2020` Chapter 10 for the probabilistic reading. The digits
> data ships with scikit-learn {cite}`pedregosa2011sklearn` — no download.

## Theory in brief

### The data matrix, centred

Data arrives as $X \in \mathbb{R}^{n\times d}$: $n$ samples in rows, $d$
features in columns. PCA begins by subtracting each column's mean,

```{math}
:label: eq-pca-centre
X_c = X - \mathbf{1}\bar{\mathbf{x}}^{\top},
\qquad \bar{\mathbf{x}} = \tfrac{1}{n}X^{\top}\mathbf{1},
```

and the centring is not a nicety — Exercise 5 measures what happens without
it. The **sample covariance** is then the Gram matrix

```{math}
:label: eq-pca-covariance
C = \frac{X_c^{\top}X_c}{n-1} \in \mathbb{R}^{d\times d},
```

symmetric positive semidefinite by
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb), so the spectral
theorem applies in full: $C = Q\Lambda Q^{\top}$ with orthonormal $Q$ and
$\lambda_1 \ge \dots \ge \lambda_d \ge 0$.

### PCA is the SVD

Write the economy SVD $X_c = U\Sigma V^{\top}$. Then
$C = V\bigl(\Sigma^2/(n-1)\bigr)V^{\top}$, so

```{math}
:label: eq-pca-svd
\mathbf{q}_i = \mathbf{v}_i,
\qquad
\lambda_i = \frac{\sigma_i^2}{n-1} :
```

the principal directions are the right singular vectors of the centred data
and the variances are its scaled squared singular values. Computing PCA
through `eigh(C)` and through `svd(X_c)` must therefore agree exactly — and
the second is how it is *done*, because forming $C$ squares the condition
number, the sin measured in
[§2.3](../02-orthogonality/least-squares-four-ways.ipynb).

The **scores** (coordinates of each sample in the principal basis) and the
rank-$k$ **reconstruction** are

```{math}
:label: eq-pca-scores
Z = X_cV = U\Sigma,
\qquad
X_c^{(k)} = Z_{[:, :k]}V_{[:, :k]}^{\top} = U_k\Sigma_kV_k^{\top},
```

and the second equality says the reconstruction is *exactly* the truncated SVD
of [§4.2](low-rank-eckart-young.ipynb), so Eckart–Young prices it:

```{math}
:label: eq-pca-eckart
\bigl\|X_c - X_c^{(k)}\bigr\|_F^2
  = \sum_{i>k}\sigma_i^2
  = (n-1)\sum_{i>k}\lambda_i .
```

The **explained-variance ratio** of component $i$ is
$\lambda_i/\sum_j\lambda_j$ — Frobenius energy fractions, renamed.

### Whitening

Whitening seeks $W$ with $\operatorname{Cov}(X_cW) = I$: coordinates that are
uncorrelated with unit variance. Two standard choices,

```{math}
:label: eq-pca-whiten
W_{\text{PCA}} = Q\Lambda^{-1/2},
\qquad
W_{\text{ZCA}} = Q\Lambda^{-1/2}Q^{\top} = C^{-1/2},
```

both work, and they differ by the rotation $Q^{\top}$: ZCA is the *symmetric*
inverse square root — the matrix-function $C^{-1/2}$ — and is the whitener
that stays closest to the identity. This is
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb)'s sampling trick
run in reverse: there $L$ manufactured correlation from white noise; here
$C^{-1/2}$ removes it.

### PCA is not regression

On a 2-D cloud, the regression line minimises **vertical** residuals
$\sum(y_i - a x_i)^2$; the first principal axis minimises **perpendicular**
ones — total least squares. Different objectives, different lines, and the
perpendicular residual has a closed form: its sum of squares is
$(n-1)\lambda_{\min}$, the variance PCA discards.

---
## Setup

Data only: the two-dimensional cloud and the digits. PCA-via-SVD — the
method this notebook crowns — is built in Exercise 1, where the crowning
is the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA as SKPCA

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# The 2-D worked example: 400 points from the correlated Gaussian of 3.3's
# sampling trick, with covariance [[3, 1.4], [1.4, 1]] and mean (2, -1).
N2 = 400
SIGMA2 = np.array([[3.0, 1.4], [1.4, 1.0]])
MU2 = np.array([2.0, -1.0])
X2 = (np.linalg.cholesky(SIGMA2) @ rng.standard_normal((2, N2))).T + MU2

# The real data: scikit-learn's bundled 1797 handwritten digits, 64 features.
DIGITS = load_digits()
XD = DIGITS.data.astype(float)          # (1797, 64)
YD = DIGITS.target

## Exercise 1: Two routes, one answer, and why one of them is the method

{eq}`eq-pca-svd` says `eigh` on the covariance and `svd` on the centred data
are the same computation. This exercise runs both on the 400-point cloud
`X2`, drawn from the correlated Gaussian with
$\Sigma = \left[\begin{smallmatrix}3&1.4\\1.4&1\end{smallmatrix}\right]$ and
mean $(2, -1)$ by the Cholesky trick of
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb).

**Part a)** Centre with `Xc = X2 - X2.mean(axis=0)` and form the covariance
$C = X_c^{\top}X_c/(n-1)$ per {eq}`eq-pca-covariance`. Confirm it matches
`np.cov(X2.T)` to $10^{-12}$ and estimates the true $\Sigma$ to within $15\%$
per entry — it is a 400-sample estimate, not the truth.

**Part b)** Compute `lam_e, Q = np.linalg.eigh(C)` (ascending, per
[§3.2](../03-eigenvalues/spectral-theorem.ipynb)) and the SVD route via
your own `pca_svd(X2)` — write it: centre, economy SVD, variances
$\sigma_i^2/(n-1)$, components as rows of $V^{\top}$. Confirm the
variances agree to a relative $10^{-10}$ after
ordering, and confirm the *directions* agree as **projectors**
$\mathbf{v}\mathbf{v}^{\top}$ to $10^{-12}$ — the sign of each vector is the
library's choice, as it has been since
[§3.1](../03-eigenvalues/eigenvalues-diagonalization.ipynb).

**Write `pca_svd` yourself** — the implementation is the lesson.

**Part c)** Confirm $\operatorname{tr}C = \sum_i\lambda_i$ to $10^{-12}$: the
total variance is basis-independent, which is what makes "explained variance"
a meaningful fraction.

**Part d)** Confirm the variance interpretation directly: the sample variance
of the projections $X_c\mathbf{v}_1$ equals $\lambda_1$ to a relative
$10^{-10}$, and likewise for $\mathbf{v}_2$ — and that $\mathbf{v}_1$ beats
$10^{3}$ random unit directions, none of whose projection variances exceeds
$\lambda_1$. That maximisation is the Rayleigh-quotient characterisation of
[§3.2](../03-eigenvalues/spectral-theorem.ipynb), applied to $C$.

**Part e)** Draw the cloud with both principal axes through the sample mean,
each scaled to $2\sqrt{\lambda_i}$ — the $2\sigma$ extent along its
direction.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

The two routes are compared as projectors, not vectors — sign freedom, as
always — and the maximisation claim is tested against random challengers the
same way Courant–Fischer was: by counting violations, of which there are none.

In [ ]:
validate.below(
    cov_gap, 1e-12,
    "C = Xc^T Xc/(n-1) is np.cov exactly (Eq. 2)",
)
validate.close(
    lam_e[::-1], lam_s,
    "eigh on C and svd on Xc give the same variances (Eq. 3)",
    rtol=1e-10, atol=0.0,
)
validate.below(
    proj_gap, 1e-12,
    "and the same directions, compared as projectors (Eq. 3)",
    "the sign of each vector is the library's choice; the projector is not",
)
validate.below(
    tr_gap, 1e-12,
    "total variance is basis-independent: tr C = sum lam",
)
validate.check(
    np.abs(var_proj - lam_s).max() / lam_s[0] < 1e-10
    and bool((var_rand <= lam_s[0] + 1e-12).all()),
    "lam_i IS the variance along v_i, and no direction of 1000 beats v_1",
    f"projection variances {np.round(var_proj, 4)} against lam "
    f"{np.round(lam_s, 4)}; best random direction reaches "
    f"{var_rand.max()/lam_s[0]:.4f} of lam_1 — the Rayleigh quotient of 3.2, "
    "applied to C",
)

## Exercise 2: The digits: scores, scree, and Eckart–Young in costume

The real data is scikit-learn's 1797 handwritten digits: $n = 1797$ samples,
$d = 64$ features (the pixels of an $8\times8$ image), bundled offline. This
exercise runs the full PCA pipeline on it and checks each statistical claim
against the linear-algebra identity it renames.

**Part a)** Run `pca_svd(XD)` and form the scores $Z = X_cV$ per
{eq}`eq-pca-scores`. Confirm $Z = U\Sigma$ to $10^{-10}$ (computing $U\Sigma$
from the SVD directly), and confirm the scores are **uncorrelated**: the
off-diagonal entries of $\operatorname{Cov}(Z)$ are below
$10^{-10}\lambda_1$, which is the orthogonality of $V$ speaking.

**Part b)** Compute the explained-variance ratios $\lambda_i/\sum\lambda_j$.
The first two are $14.9\%$ and $13.6\%$ — two components carry $28.5\%$ of
64-dimensional pixel variance. Confirm the ratios match
`sklearn.decomposition.PCA(n_components=10).fit(XD).explained_variance_ratio_`
to $10^{-8}$. sklearn is not consulted for the answer; it is confirmed to
*be* this computation.

**Part c)** Confirm {eq}`eq-pca-eckart` at $k = 5, 10, 20$: the squared
Frobenius reconstruction error equals $\sum_{i>k}\sigma_i^2$ to a relative
$10^{-9}$. This is Eckart–Young, and it prices every choice of $k$ from the
scree plot alone before any reconstruction is formed — exactly as in
[§4.2](low-rank-eckart-young.ipynb).

**Part d)** Report how many components reach $90\%$ cumulative explained
variance (21), and confirm the cumulative curve is monotone with the last
value $1$ to $10^{-12}$.

**Part e)** Draw the scree plot with its cumulative curve, and the 1797
digits in the plane of their first two scores, coloured by label — the
classic picture, with the classes visibly organised by two numbers per image.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

The sklearn comparison is the sociological check — an industrial-standard
implementation agreeing to $10^{-8}$ says PCA *is* this SVD, not a cousin of
it — while the reconstruction identity is the mathematical one, Eckart–Young
priced from the spectrum exactly as in the previous notebook.

In [ ]:
validate.check(
    z_gap < 1e-10 and offdiag < 1e-10 * lam_d[0],
    "the scores are U Sigma, and they are exactly uncorrelated (Eq. 4)",
    f"Z vs U Sigma to {z_gap:.1e}; worst score covariance off-diagonal "
    f"{offdiag:.1e} — the orthogonality of V, read statistically",
)
validate.below(
    sk_gap, 1e-8,
    "sklearn's explained-variance ratios ARE these, to 1e-8",
    "sklearn.decomposition.PCA computes the same centred SVD",
)
validate.below(
    max(ey_rel), 1e-9,
    "the k-component reconstruction error is the tail energy (Eq. 5)",
    "Eckart-Young in statistical costume, at k = 5, 10, 20",
)
validate.check(
    k90 == 21 and bool(np.all(np.diff(cum) >= -1e-15))
    and abs(cum[-1] - 1.0) < 1e-12,
    "90% of the pixel variance lives in 21 of 64 components",
    f"cumulative curve monotone, ending at {cum[-1]:.12f}",
)

## Exercise 3: Eigendigits, and reconstruction as a budget

The principal directions of image data are themselves images, and looking at
them explains what the scores measure. This exercise draws the first sixteen
— the **eigendigits** — and watches a single digit reassemble as components
are added back per {eq}`eq-pca-scores`.

**Part a)** Reshape the first sixteen rows of $V^{\top}$ to $8\times8$ and
draw them. Confirm each has unit norm to $10^{-12}$ and that the sixteen are
mutually orthogonal to $10^{-12}$ — they are rows of an orthogonal matrix,
whatever else they look like.

**Part b)** Reconstruct digit number 3 (the same "3" as
[§4.2](low-rank-eckart-young.ipynb)) at $k = 5, 10, 20$ via
$\bar{\mathbf{x}} + \sum_{i\le k}z_i\mathbf{v}_i$, and report the relative
error against the original at each $k$. Confirm it decreases and that the
$k = 64$ reconstruction is exact to $10^{-10}$.

**Part c)** Confirm the mean matters: the $k = 0$ "reconstruction" — the mean
image alone — already carries the digit-shaped blur that all 1797 samples
share, with relative error $57\%$ against this digit; the components only
encode the *departure* from it.

**Part d)** Draw the reconstruction sequence beside the original.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The orthogonality check is what separates "sixteen suggestive pictures" from
"rows of an orthogonal matrix": the eigendigits are constrained objects, and
the constraint is verified rather than admired.

In [ ]:
validate.check(
    unit_gap < 1e-12 and orth_gap < 1e-12,
    "the sixteen eigendigits are orthonormal, whatever they look like",
    f"unit norms to {unit_gap:.1e}, mutual orthogonality to {orth_gap:.1e}",
)
validate.check(
    rec_err[5] > rec_err[10] > rec_err[20] and rec_err[64] < 1e-10,
    "the reconstruction improves monotonically and is exact at k = d (Eq. 4)",
    f"errors {rec_err[5]:.3f} > {rec_err[10]:.3f} > {rec_err[20]:.3f}, and "
    f"{rec_err[64]:.1e} with all 64 components",
)
validate.check(
    0.30 < rec_err[0] < 0.60,
    "while the mean alone is already within 57% of this digit",
    f"relative error {rec_err[0]:.3f} at k = 0: the components encode only the "
    "departure from what every sample shares",
)

## Exercise 4: Whitening: PCA run backwards

[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb) *built* correlation:
$\mathbf{x} = L\mathbf{z}$ gave white noise the covariance $LL^{\top}$.
Whitening inverts that. This exercise applies both whiteners of
{eq}`eq-pca-whiten` to the 2-D cloud and verifies what each does and what
distinguishes them.

**Part a)** Build $W_{\text{PCA}} = Q\Lambda^{-1/2}$ and
$W_{\text{ZCA}} = Q\Lambda^{-1/2}Q^{\top}$ from `eigh(C2)`. Confirm both give
$\operatorname{Cov}(X_cW) = I_2$ to $10^{-10}$.

**Part b)** Confirm $W_{\text{ZCA}} = C^{-1/2}$ as a matrix function: its
square times $C$ is $I$ to $10^{-12}$, and it is symmetric to $10^{-14}$,
while $W_{\text{PCA}}$ is not symmetric. The two whiteners differ by the
rotation $Q^{\top}$, and rotating white data leaves it white — confirm that
too, by checking $\operatorname{Cov}$ is still $I$ after applying a random
rotation to the ZCA output.

**Part c)** Confirm ZCA is the whitener nearest the identity:
$\|W_{\text{ZCA}} - I\|_F < \|W_{\text{PCA}} - I\|_F$, which is why it is the
choice when the coordinates mean something.

**Part d)** Close the loop with [§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb):
whiten the cloud, then re-colour it with the *true* $\Sigma$'s Cholesky
factor, and confirm the recoloured covariance matches $\Sigma$ to the
sampling accuracy of Part a) of Exercise 1 ($15\%$) — manufacture and removal
of correlation are the same operation with the triangle inverted.

**Part e)** Draw the cloud, its PCA-whitened version, and its ZCA-whitened
version on equal axes: both whitened clouds are round, and the ZCA one is
visibly the *least rotated* relative to the original.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

The matrix-function reading is the check with content: $W_{\text{ZCA}}$
squaring against $C$ to the identity certifies it as $C^{-1/2}$, an object
[§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb) will construct
in general — and the closed loop with the colouring trick ties the two
notebooks into one statement.

In [ ]:
validate.check(
    g_pca < 1e-10 and g_zca < 1e-10,
    "both whiteners produce identity covariance (Eq. 6)",
    f"PCA route {g_pca:.1e}, ZCA {g_zca:.1e} from I_2",
)
validate.check(
    inv_sqrt < 1e-12 and sym_zca < 1e-14 and sym_pca > 0.1,
    "ZCA is the SYMMETRIC inverse square root C^{-1/2}; the PCA whitener is not",
    f"W^2 C = I to {inv_sqrt:.1e}, symmetry {sym_zca:.1e} against the PCA "
    f"whitener's {sym_pca:.3f} asymmetry",
)
validate.below(
    g_rot, 1e-10,
    "and whiteness survives rotation, which is why W is only unique up to one",
)
validate.check(
    d_zca < d_pca,
    "ZCA is the whitener nearest the identity",
    f"||W - I||_F: {d_zca:.4f} against {d_pca:.4f}",
)
validate.below(
    recol_rel, 0.20,
    "whiten-then-recolour reproduces the true Sigma to sampling accuracy",
    "colouring (3.3) and whitening are one operation, inverted",
)

## Exercise 5: The uncentred trap, and the line PCA actually fits

Two closing measurements, each guarding a standard mistake.

**Skipping the centring.** Run the SVD on the *raw* digits matrix and its
first right singular vector is not a principal direction at all: it points at
the **mean**. The rank-one term $\sigma_1\mathbf{u}_1\mathbf{v}_1^{\top}$ of
an uncentred data matrix is busy representing $\mathbf{1}\bar{\mathbf{x}}^{\top}$,
and every "component" after it is contaminated by the leftovers.

**PCA is not regression.** On the 2-D cloud, the regression of $x_2$ on
$x_1$ minimises vertical residuals; the first principal axis minimises
perpendicular ones. Two objectives, two different slopes — and the
perpendicular one comes with the exact identity that its residual sum of
squares is $(n-1)\lambda_2$, the variance PCA discards.

**Part a)** Compute the SVD of the raw `XD` and report
$|\cos\theta|$ between $\mathbf{v}_1^{\text{raw}}$ and the mean image
$\bar{\mathbf{x}}$: it is $0.99998$. Report also
$\sigma_1^{\text{raw}} = 2193$ against the centred $\sigma_1 = 567$: the
"leading component" of uncentred data is mostly the mean, amplified
$3.9\times$.

**Part b)** Confirm the centred $\mathbf{v}_1$ is *not* the mean direction:
$|\cos\theta| < 0.6$ there. Centring is what makes the components describe
variation rather than location.

**Part c)** On the cloud, compute the regression slope
$\hat{a} = \sum x_1x_2 / \sum x_1^2$ (centred data) and the PCA slope from
$\mathbf{v}_1$. They differ: $0.496$ against $0.546$ — a $10\%$ disagreement
on the same 400 points. Confirm the two are distinct by more than $5\%$ and
that *neither* is wrong: each minimises exactly its own objective, checked in
Part d).

**Part d)** Verify both optimality claims by perturbation: over 200 slopes
in a $\pm20\%$ bracket around each optimum, no candidate beats the regression
slope on vertical residuals, and none beats the PCA slope on perpendicular
ones. And confirm the closed form: the perpendicular residual sum of squares
at the PCA slope equals $(n-1)\lambda_2$ to a relative $10^{-12}$.

**Part e)** Draw the cloud with both lines, and the two residual objectives
as curves over the slope bracket, each minimised at its own line.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
The scores of Exercise 2 used all 1797 samples to *build* the basis and then
projected the same samples onto it. Ask your assistant for a
`pca_transform(X_train, X_new, k)` that fits the mean and components on one
set and projects another — the split every real pipeline uses. Then check it
against the mathematics rather than against sklearn's output: verify (i) on
`X_new = X_train` it reproduces Exercise 2's scores to $10^{-10}$, (ii) the
*train*-set reconstruction error at rank $k$ still equals the tail energy of
Eq. 5 exactly, while (iii) the *held-out* reconstruction error is strictly
larger for every $k$ tested — Eckart–Young optimises the matrix it was given,
and the gap between (ii) and (iii) is the honest definition of overfitting a
basis. The check is yours.
```

### Validation 5

Both optimality claims are tested by perturbation, because "minimises" is a
claim about a neighbourhood, not a formula — and the trap checks are gated on
the *measured* cosines, which is what makes "always centre first" an observed
fact about this data rather than advice.

In [ ]:
validate.check(
    cos_raw > 0.999 and s_raw[0] / sd[0] > 3.0,
    "uncentred PCA's first component points at the MEAN (Eq. 1)",
    f"|cos| = {cos_raw:.5f} with sigma_1 inflated {s_raw[0]/sd[0]:.1f}x: the "
    "leading rank-one term is representing location, not variation",
)
validate.check(
    cos_cent < 0.6,
    "while the centred first component does not",
    f"|cos| = {cos_cent:.3f}: centring is what points the basis at the variance",
)
validate.check(
    abs(slope_pca - slope_reg) / abs(slope_reg) > 0.05,
    "the PCA line and the regression line are measurably different lines",
    f"slopes {slope_pca:.4f} against {slope_reg:.4f} on the same 400 points: "
    "perpendicular and vertical residuals are different objectives",
)
validate.check(
    reg_wins and pca_wins,
    "and each is optimal for exactly its own objective, by perturbation",
    "200 candidate slopes in a +/-20% bracket: none beats regression on "
    "vertical SS, none beats the principal axis on perpendicular SS",
)
validate.below(
    perp_identity, 1e-12,
    "with the PCA line's perpendicular SS equal to (n-1) lam_2 exactly",
    "the discarded variance IS the residual",
)

---
## Notebook summary

**PCA is the SVD of the centred data matrix.** On the 2-D cloud, `eigh` on
$C$ and `svd` on $X_c$ agreed to $10^{-16}$ as projectors with variances
matching to $10^{-15}$ relative; $\operatorname{tr}C = \sum\lambda_i$ to
$10^{-15}$; and $\lambda_i$ *is* the projection variance along
$\mathbf{v}_i$, with none of $10^{3}$ random directions beating
$\mathbf{v}_1$ — the Rayleigh quotient of
[§3.2](../03-eigenvalues/spectral-theorem.ipynb) in statistical clothes.

**The statistical vocabulary is renamed linear algebra, verified item by
item.** On the 1797 digits: scores $= U\Sigma$ and exactly uncorrelated;
explained-variance ratios matching `sklearn` to $3\times10^{-16}$;
reconstruction error equal to the tail energy at $k = 5, 10, 20$ to
$10^{-15}$ relative — Eckart–Young pricing every $k$ from the scree plot
before anything is formed. $90\%$ of the pixel variance lives in 21 of 64
components, and the sixteen eigendigits are orthonormal to $10^{-13}$
whatever they look like.

**Whitening inverts the colouring trick.** Both whiteners produced identity
covariance to $10^{-15}$; $W_{\text{ZCA}}$ certified itself as the symmetric
$C^{-1/2}$ (its square times $C$ is $I$ to $10^{-13}$) and is the whitener
nearest the identity, while whiteness survived a random rotation — the
non-uniqueness made visible. Whiten-then-recolour reproduced the true
$\Sigma$ to sampling accuracy, closing the loop with
[§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb).

**Two traps, measured.** Uncentred, the first singular vector of the digits
points at the mean with $|\cos| = 0.99998$ and $\sigma_1$ inflated
$3.9\times$ — the leading "component" is location, not variation. And the
PCA line is not the regression line: slopes $0.546$ against $0.496$ on the
same cloud, each optimal for exactly its own residual by a 200-candidate
perturbation test, with the perpendicular residual equal to $(n-1)\lambda_2$
to $10^{-13}$ relative.

**Methods introduced.** `pca_svd` (centring + economy SVD, never forming
$C$), scores and loadings, explained-variance and scree plots,
`sklearn.decomposition.PCA` as a cross-check, eigendigit galleries,
PCA and ZCA whitening, whiten-recolour round trips, and the
vertical-versus-perpendicular residual comparison.

## Outlook

- **When even one pass over the data is too much.** The digits SVD was
  instant; at web scale the centred data matrix does not fit anywhere.
  [§4.4](randomized-svd-sketching.ipynb) gets the leading components from a
  handful of matrix–vector products, with a failure probability one can set.
- **What pure noise looks like.** The scree plot's tail was not zero, and
  deciding where signal ends needs a null model: the Marchenko–Pastur law for
  the singular values of pure noise, against which a component either stands
  out or does not.
- **$C^{-1/2}$ in general.** ZCA needed the inverse square root of one
  well-conditioned $2\times2$; the matrix-function machinery of
  [§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb) builds $f(A)$
  for general $f$, and the whitening here is its simplest useful case.
- **Centring as a projection.** Subtracting the mean is multiplication by
  $I - \tfrac{1}{n}\mathbf{1}\mathbf{1}^{\top}$, an orthogonal projector of
  rank $n-1$ — which is why uncentred PCA wastes its first component on
  $\mathbf{1}$: that direction was never projected out.
  [§6.1](../06-structure/graphs-laplacian.ipynb) meets the same projector as
  the null space of every graph Laplacian.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()